In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Configurações
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid")

import warnings
warnings.filterwarnings("ignore")

# 1. Importing Dataset

In [2]:
df = pd.read_csv("../data/raw/Superstore.csv", encoding="latin1")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


## Data Understanding

The dataset contains transactional sales records from a fictional Superstore.
Each row represents an order line, including information about customer segment,
product category, sales, profit, discount, shipping mode, region, and order dates.

The purpose of this analysis is to identify the main drivers of sales and profitability,
with special attention to discounts, product categories, regional performance,
and shipping behavior.

# 2. Dataset Preliminary Diagnosis

In [3]:
df.shape

(9994, 21)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Row ID,"9,994.00",NaN,NaN,NaN,"4,997.50","2,885.16",1.00,"2,499.25","4,997.50","7,495.75","9,994.00"
Order ID,9994,5009,CA-2017-100111,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,9994,1237,9/5/2016,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Date,9994,1334,12/16/2015,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Mode,9994,4,Standard Class,5968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,9994,793,WB-21850,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,9994,793,William Brown,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,9994,3,Consumer,5191,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,9994,1,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,9994,531,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.isna().sum().sort_values(ascending=False)


Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

# 3. Column Naming Standardization

In [8]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

# 4. Date Processing and Formatting

In [9]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")

In [10]:
df[["order_date", "ship_date"]].isna().sum()

order_date    0
ship_date     0
dtype: int64

In [11]:
df["days_to_ship"] = (df["ship_date"] - df["order_date"]).dt.days

In [12]:
df[df["days_to_ship"] < 0]

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,days_to_ship


In [14]:
df = df[df["days_to_ship"] >= 0]

# 5. Missing Data Handling

In [15]:
missing = (
    df.isna()
    .sum()
    .to_frame("missing_count")
)

missing["missing_pct"] = missing["missing_count"] / len(df) * 100
missing.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
row_id,0,0.00
order_id,0,0.00
order_date,0,0.00
ship_date,0,0.00
ship_mode,0,0.00
customer_id,0,0.00
customer_name,0,0.00
segment,0,0.00
country,0,0.00
city,0,0.00


In [16]:
df = df.drop(columns=["postal_code"], errors="ignore")

### Missing Values Treatment

The missing value analysis showed that postal code was not relevant for the business questions
addressed in this project. Since the analysis focuses on profitability, categories, regions,
shipping performance, and discount behavior, this column was removed from the analytical dataset.

# 6. Deduplication

In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
df = df.drop_duplicates()

# 7. Standardizing Categorical Variables

In [ ]:
categorical_cols = [
    "ship_mode", "segment", "country", "city", "state",
    "region", "category", "sub_category"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

In [ ]:
for col in ["ship_mode", "segment", "region", "category"]:
    print(col)
    print(df[col].unique())
    print("-" * 50)

# 8. Feature Engineering

## 8.1 Profit margin

In [ ]:
df["profit_margin"] = df["profit"] / df["sales"]

In [ ]:
df["profit_margin"] = np.where(
    df["sales"] > 0,
    df["profit"] / df["sales"],
    np.nan
)